# Post process data extracted using LLMs
1. Load data
2. Convert to correct format (numeric for numerical columns)
3. Eventually explode lists for geocoding?
4. Geocoding
5. Sanity checks

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import geopy as gpy
import time
import itertools
import regex as re
from matplotlib import pyplot as plt
from src.data import *
from src.text_processing_functions import *
from src.plot_functions import *
from src.post_process_functions import *
from src.geocoding import *
from src.hazard_def import *
from src.impact_def import *
from src.sanity_checks import *



[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/lseverino/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /Users/lseverino/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/lseverino/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
#load data (model)

#res_savename = 'llm_response_impact_labelled_reports_test_multiprompt_continue_v050925_21rep_meta-llama_llama-4-scout-17b-16e-instruct.csv'
#response_df = pd.read_csv(DATA_OUT_LLMS /res_savename)

#load data (labelled)
res_savename = "labelled_reports_impacts_all_v080925.csv"
response_df = pd.read_csv(DATA_LABELLED / res_savename)


In [3]:
response_df

,reportDate,impactSubtype,impactValue,impactUnit,impactValuePrecision,impactValueMin,impactValueMax,annotation,country,location,startYear,startMonth,startDay,endYear,endMonth,endDay,hazards,appealCode,comments
0,2024-08-06,Residential Buildings,55.0,houses damaged,exact,NaN,NaN,['Barbados was spared a direct hit from the hu...,['Barbados'],NaN,2024.0,6.0,25.0,NaN,NaN,NaN,['Tropical storm'],MDRS2001,NaN
1,2024-08-06,Agricultural Infrastructure,200.0,vessels,exact,NaN,NaN,['Barbados was spared a direct hit from the hu...,['Barbados'],NaN,2024.0,6.0,25.0,NaN,NaN,NaN,['Tropical storm'],MDRS2001,NaN
2,2024-08-06,AOther Economic and Livelihood Impacts,NaN,NaN,NaN,NaN,NaN,['Barbados was spared a direct hit from the hu...,['Barbados'],NaN,2024.0,6.0,25.0,NaN,NaN,NaN,['Tropical storm'],MDRS2001,NaN
3,2024-08-06,Displaced People,NaN,people,approx,1600.0,NaN,"['On July 1, the hurricane made landfall in Gr...",['Grenada'],NaN,2024.0,7.0,1.0,NaN,NaN,NaN,['Tropical storm'],MDRS2001,NaN
4,2024-08-06,Other Infrastructure impact,98.0,% buildings damaged,exact,NaN,NaN,"['On July 1, the hurricane made landfall in Gr...",['Grenada'],"['Carriacou islands', 'Petit Martinique island']",2024.0,7.0,1.0,NaN,NaN,NaN,['Tropical storm'],MDRS2001,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
487,2017-05-31,Residential Buildings,11356.0,houses damaged,exact,NaN,NaN,['Situation analysis Description of the disast...,['Philippines'],NaN,2016.0,10.0,16.0,2016.0,10.0,17.0,['Tropical storm'],MDRPH021,NaN
488,2017-05-31,Residential Buildings,1421.0,houses destroyed,exact,NaN,NaN,['Situation analysis Description of the disast...,['Philippines'],NaN,2016.0,10.0,16.0,2016.0,10.0,17.0,['Tropical storm'],MDRPH021,NaN
489,2017-05-31,Crop Production and Forestry,NaN,CHF,approx,74000000.0,NaN,['Situation analysis Description of the disast...,['Philippines'],NaN,2016.0,10.0,16.0,2016.0,10.0,17.0,['Tropical storm'],MDRPH021,NaN
490,2017-05-31,Infrastructure,NaN,CHF,approx,46000000.0,NaN,['Situation analysis Description of the disast...,['Philippines'],NaN,2016.0,10.0,16.0,2016.0,10.0,17.0,['Tropical storm'],MDRPH021,NaN


In [4]:
#get rid of nans
response_df_proc = cp.deepcopy(response_df)

response_df_proc = response_df_proc.dropna(subset=["nathaz_text"]) if "nathaz_text" in response_df_proc.columns else response_df_proc

In [5]:
#process impactValue
response_df_proc[["impactValue", "impactValueMin", "impactValueMax"]] = response_df_proc.apply(parse_impact_value_precision, axis=1)

/Users/lseverino/Documents/PhD/Projects/Como/como_project4/src/post_process_functions.py:192: RuntimeWarning: All-NaN slice encountered
  min_value = np.nanmin(all_values)
/Users/lseverino/Documents/PhD/Projects/Como/como_project4/src/post_process_functions.py:193: RuntimeWarning: All-NaN slice encountered
  max_value = np.nanmax(all_values)


In [6]:
#convert numerical columns
num_cols = ["impactValue", "impactValueMin", "impactValueMax","startYear", "startMonth", "startDay", "endYear", "endMonth", "endDay"]
list_cols = ["country","location", "hazards", "valueAnnotation", "locationAnnotation", "dateAnnotation", "hazardsAnnotation", "annotation"]
list_cols = [key for key in list_cols if key in response_df_proc.columns]
response_df_proc = format_output(response_df_proc, num_cols=num_cols, list_cols=list_cols)



In [7]:
response_df_proc[["impactValue", "impactValueMin", "impactValueMax", "annotation"]]

,impactValue,impactValueMin,impactValueMax,annotation
0,55.0,NaN,NaN,[Barbados was spared a direct hit from the hur...
1,200.0,NaN,NaN,[Barbados was spared a direct hit from the hur...
2,NaN,NaN,NaN,[Barbados was spared a direct hit from the hur...
3,1600.0,1600.0,NaN,"[On July 1, the hurricane made landfall in Gre..."
4,98.0,NaN,NaN,"[On July 1, the hurricane made landfall in Gre..."
...,...,...,...,...
487,11356.0,NaN,NaN,[['Situation analysis Description of the disas...
488,1421.0,NaN,NaN,[['Situation analysis Description of the disas...
489,74000000.0,74000000.0,NaN,[['Situation analysis Description of the disas...
490,46000000.0,46000000.0,NaN,[['Situation analysis Description of the disas...


In [8]:
#add iso3
response_df_proc["country_iso3"] = response_df_proc["country"].apply(list_country_name_to_iso3)
response_df_proc["country_iso3_kw"] = response_df_proc["country_kw"].apply(list_country_name_to_iso3) if "country_kw" in response_df_proc.columns else None

In [9]:
#format units
from spacy.lang.en import English
from spacy.lang.punctuation import TOKENIZER_PREFIXES, TOKENIZER_SUFFIXES, TOKENIZER_INFIXES
from spacy.lang.en import TOKENIZER_EXCEPTIONS
from spacy.tokenizer import Tokenizer
from spacy.util import compile_prefix_regex, compile_suffix_regex, compile_infix_regex

## Post process
1. Reclassify hazards
2. Reclassify impactSubtypes
3. Reclassify units

In [10]:
impactSubtype_list

['Affected People',
 'Injured People',
 'Displaced People',
 'Homeless People',
 'Missing People',
 'Human Deaths',
 'Human Health and Wellbeing',
 'Infected and Ill People',
 'Road Infrastructure',
 'Other Transportation Infrastructure',
 'Water, Sanitation, and Hygiene Infrastructure',
 'Healthcare Infrastructure',
 'IT and Communication Infrastructure',
 'Residential Buildings',
 'Informal settlements',
 'Education Infrastructure',
 'Power and Energy Production Infrastructure',
 'Agriculture Infrastructure',
 'Crop Production and Forestry',
 'Affected Livestock and Animals',
 'Other Economic and Livelihood Impacts',
 'Recreation, Tourism, and Culture',
 'Access to Healthcare',
 'Access to transport and Mobility',
 'Water Quality and Availability',
 'Access to Education',
 'Access to Power and Energy',
 'Access to Food',
 'Access to Water, Sanitation, and Hygiene',
 'Other Human Impacts',
 'Other Infrastructure Impacts',
 'Other Agricultural Impacts',
 'Other Service Access Impacts']

In [11]:
#reclassify impacType
response_df_proc["impactSubtype_orig"] = response_df_proc["impactSubtype"]
response_df_proc["impactSubtype"] = response_df_proc.apply(reclassify_impact_subtype, allowed_impact_types=impactSubtype_list, impact_kw_reclass=impact_kw_reclass, axis=1)


In [12]:
response_df_proc[["impactSubtype", "impactSubtype_orig"]].where(response_df_proc["impactSubtype"]=="Unknown").dropna(how="all")

,impactSubtype,impactSubtype_orig
490,Unknown,Infrastructure


In [14]:
#reclassify hazard
response_df_proc["hazards_orig"] = response_df_proc["hazards"]
response_df_proc["hazards"] = response_df_proc.apply(reclassify_hazard, hazard_kw_reclass=hazard_kw_reclass, axis=1)
explode_lists(response_df_proc).hazards.value_counts()

/Users/lseverino/Documents/PhD/Projects/Como/como_project4/src/post_process_functions.py:117: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  repeat_counts = df[list_columns].applymap(len).max(axis=1)


hazards
Flood                       913
Mass movement               254
Tropical storm              204
Other storm                  84
Epidemics                    39
Convective storm             21
Drought                      18
Conflict                      9
Extreme warm temperature      3
Name: count, dtype: int64

In [15]:
wrong_haz = explode_lists(response_df_proc).copy()
wrong_haz = wrong_haz[wrong_haz["hazards"] == "Unknown"]
wrong_haz["hazards"]

/Users/lseverino/Documents/PhD/Projects/Como/como_project4/src/post_process_functions.py:117: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  repeat_counts = df[list_columns].applymap(len).max(axis=1)


Series([], Name: hazards, dtype: object)

In [16]:
impactSubtype_list

['Affected People',
 'Injured People',
 'Displaced People',
 'Homeless People',
 'Missing People',
 'Human Deaths',
 'Human Health and Wellbeing',
 'Infected and Ill People',
 'Road Infrastructure',
 'Other Transportation Infrastructure',
 'Water, Sanitation, and Hygiene Infrastructure',
 'Healthcare Infrastructure',
 'IT and Communication Infrastructure',
 'Residential Buildings',
 'Informal settlements',
 'Education Infrastructure',
 'Power and Energy Production Infrastructure',
 'Agriculture Infrastructure',
 'Crop Production and Forestry',
 'Affected Livestock and Animals',
 'Other Economic and Livelihood Impacts',
 'Recreation, Tourism, and Culture',
 'Access to Healthcare',
 'Access to transport and Mobility',
 'Water Quality and Availability',
 'Access to Education',
 'Access to Power and Energy',
 'Access to Food',
 'Access to Water, Sanitation, and Hygiene',
 'Other Human Impacts',
 'Other Infrastructure Impacts',
 'Other Agricultural Impacts',
 'Other Service Access Impacts']

### Units reclassification
1. Standardize units i.e. metric units to SI or common units.
2. Determine unit typology (e.g. distance, surface, weight, percent) and special units (money, deaths)
3. Convert non-metric units e.g. households to people, USD to CHF
4. Standardize non-metric units i.e. person, children to people, hospitals to health facilities

TO DOS
1. Handle deaths so that they dont go in other categories
2. Handle targeted people
3. ~~Handle currencies~~
4. When unknown unit, try to infer it using kw for other impact subtype and reclass to other subtype if match
5. Handle damage vs destroyed houses
6. Parse correctly impact values min and maxs

In [17]:
#response_df_proc["impactValue"] = response_df_proc["impactValueOrig"]
#response_df_proc["impactUnit"] = response_df_proc["impactUnitOrig"]

In [18]:
force_unit_to_subtype = False #whether or not we want to force unit to default unit of subtype when unknown unit
#keep orig unit and value for comparison
response_df_proc["impactValueOrig"] = response_df_proc["impactValue"]
response_df_proc["impactUnitOrig"] = response_df_proc["impactUnit"]
#replace numbers in units
response_df_proc[["impactValue", "impactUnit"]] = response_df_proc.apply(replace_numbers_unit, axis=1)
#convert money
response_df_proc[["impactValue", "impactUnit"]] = response_df_proc.apply(convert_monetary_units, axis=1)
#standardize SI units
response_df_proc[["impactValue", "impactUnit"]]  = response_df_proc.apply(standardize_units, std_unit_kw_reclass=std_unit_kw_reclass, unit_mapping=unit_mapping, axis=1)
response_df_proc = response_df_proc.apply(convert_unit, unit_converter=unit_converter, axis=1)
response_df_proc["unit_type"] = response_df_proc.apply(assign_unit_type, unit_type_kw_reclass=unit_type_kw_reclass, axis=1)
response_df_proc["impactUnit"] = response_df_proc.apply(reclassify_units, unit_kw_reclass=unit_kw_reclass, default_subtype_unit=default_subtype_unit, force_unit_to_subtype=force_unit_to_subtype, axis=1)


Af is not a supported currency


In [19]:
response_df_proc[["impactSubtype","impactValueOrig", "impactValue", "impactUnitOrig", "impactUnit","annotation"]]

,impactSubtype,impactValueOrig,impactValue,impactUnitOrig,impactUnit,annotation
0,Residential Buildings,55.0,5.500000e+01,houses damaged,homes,[Barbados was spared a direct hit from the hur...
1,Agriculture Infrastructure,200.0,2.000000e+02,vessels,vessels,[Barbados was spared a direct hit from the hur...
2,Other Economic and Livelihood Impacts,NaN,NaN,NaN,nan,[Barbados was spared a direct hit from the hur...
3,Displaced People,1600.0,1.600000e+03,people,people,"[On July 1, the hurricane made landfall in Gre..."
4,Other Infrastructure Impacts,98.0,9.800000e+01,% buildings damaged,% of homes,"[On July 1, the hurricane made landfall in Gre..."
...,...,...,...,...,...,...
487,Residential Buildings,11356.0,1.135600e+04,houses damaged,homes,[['Situation analysis Description of the disas...
488,Residential Buildings,1421.0,1.421000e+03,houses destroyed,homes,[['Situation analysis Description of the disas...
489,Crop Production and Forestry,74000000.0,6.791483e+07,CHF,eur,[['Situation analysis Description of the disas...
490,Unknown,46000000.0,4.221733e+07,CHF,eur,[['Situation analysis Description of the disas...


In [20]:
#test replace numbers


In [21]:
from price_parser import Price
test_price_true1 = "1,500 USD"
parsed_price1 = Price.fromstring(test_price_true1)
print(parsed_price1)

test_price_true2 = "735 billion CHF"
parsed_price2 = Price.fromstring(test_price_true2)
print(parsed_price2)

test_price_wrong = "1,500"
parsed_price2 = Price.fromstring(test_price_wrong)
print(parsed_price2)

test_price_wrong = "1,500 PHP"
parsed_price2 = Price.fromstring(test_price_wrong)
print(parsed_price2)

Price(amount=Decimal('1500'), currency='USD')
Price(amount=Decimal('735'), currency='CHF')
Price(amount=Decimal('1500'), currency=None)
Price(amount=Decimal('1500'), currency='PHP')


In [22]:
from currency_converter import CurrencyConverter
c = CurrencyConverter()
test_conv_pass =(1500, "USD")
DEF_CUR = "EUR"
conv_price_pass = c.convert(test_conv_pass[0], test_conv_pass[1], DEF_CUR)
print(conv_price_pass)
test_conv_fail = (1500, "foo")
try:
    conv_price_fail = c.convert(test_conv_fail[0], test_conv_fail[1], DEF_CUR)
    print(conv_price_fail)
except Exception as e:
    print(e)
test_conv_fail2 = ("asda", "USD")
try:
    conv_price_fail2 = c.convert(test_conv_fail2[0], test_conv_fail2[1], DEF_CUR)
    print(conv_price_fail2)
except Exception as e:
    print(e)

1287.1117212974086
foo is not a supported currency
could not convert string to float: 'asda'


In [23]:
parsed_price1.currency

'USD'

In [24]:
unit_type = "kg"
unit = "kg of crops"
[unit_corr for unit_corr in unit_kw_reclass.keys() if re.search(unit_kw_reclass[unit_corr], unit, re.IGNORECASE)]


['crop production and forestry']

In [25]:
test_df = pd.DataFrame({
        "reportDate": ["2020-01-01", "2020-01-01", "2020-01-01"],
        "impactSubtype": ["Affected People", "Crop Production and Forestry", "Crop Production and Forestry"],
        "impactValue": [1000,1000,100],
        "impactUnit": ["families", "kg of crops","hectares of crops"]})
#replace numbers in units
test_df[["impactValue", "impactUnit"]] = test_df.apply(replace_numbers_unit, axis=1)
#convert money
test_df[["impactValue", "impactUnit"]] = test_df.apply(convert_monetary_units, axis=1)
#standardize SI units
test_df[["impactValue", "impactUnit"]]  = test_df.apply(standardize_units, std_unit_kw_reclass=std_unit_kw_reclass, unit_mapping=unit_mapping, axis=1)
test_df = test_df.apply(convert_unit, unit_converter=unit_converter, axis=1)
test_df["unit_type"] = test_df.apply(assign_unit_type, unit_type_kw_reclass=unit_type_kw_reclass, axis=1)
test_df["impactUnit"] = test_df.apply(reclassify_units, unit_kw_reclass=unit_kw_reclass, default_subtype_unit=default_subtype_unit, force_unit_to_subtype=force_unit_to_subtype, axis=1)
test_df

,reportDate,impactSubtype,impactValue,impactUnit,unit_type
0,2020-01-01,Affected People,3000.0,people,other
1,2020-01-01,Crop Production and Forestry,1000.0,kg of crop production and forestry,kg
2,2020-01-01,Crop Production and Forestry,1.0,km**2 of crop production and forestry,km**2


In [26]:
# save
savename = "post_processed_" + res_savename
response_df_proc.to_csv(DATA_OUT_PROC / savename, index=False)

In [27]:
response_df_proc[["impactValue", "impactUnit", "impactValueOrig", "impactUnitOrig", "valueAnnotation"]]

KeyError: "['valueAnnotation'] not in index"

In [28]:
response_df_proc.groupby("appealCode").count()

,reportDate,impactSubtype,impactValue,impactUnit,impactValuePrecision,impactValueMin,impactValueMax,annotation,country,location,...,endDay,hazards,comments,country_iso3,country_iso3_kw,impactSubtype_orig,hazards_orig,impactValueOrig,impactUnitOrig,unit_type
appealCode,,,,,,,,,,,,,,,,,,,,,
MDRBD022,12,12,10,12,10,2,0,12,12,12,...,1,12,0,12,0,12,12,10,10,12
MDRBJ019,10,10,6,10,10,0,1,10,10,10,...,0,10,0,10,0,10,10,6,6,10
MDRCM039,18,18,13,18,18,2,3,18,18,18,...,2,18,0,18,0,18,18,13,15,18
MDRCN006,16,16,15,16,15,5,0,16,16,16,...,13,16,0,16,0,16,16,15,16,16
MDRDZ011,20,20,9,20,20,2,1,20,20,20,...,20,20,0,20,0,20,20,9,15,20
MDRGE019,18,18,5,18,5,2,0,18,18,18,...,0,18,0,18,0,18,18,5,6,18
MDRIQ014,10,10,4,10,4,2,1,10,10,10,...,1,10,0,10,0,10,10,4,4,10
MDRKE058,48,48,40,48,40,6,0,48,48,48,...,10,48,0,48,0,48,48,40,40,48
MDRMY003,3,3,3,3,3,2,0,3,3,3,...,3,3,0,3,0,3,3,3,3,3
